# Infusion: reverse-engineering influence functions

> *Shaping model behaviour by editing training data via influence functions.*

Companion notes to {cite:t}`rosser2026infusion` — the [Infusion paper](https://jrosser.co.uk/infusion/) (DATA-FM @ ICLR 2026). The goal of this page is to:

1. Recap what an **influence function** is and why we care.
2. Derive the Infusion update — the perturbation $\delta$ you'd apply to a training document to shift a chosen behaviour.
3. Build a **toy, end-to-end, runnable** version in logistic regression, so the algebra isn't abstract.
4. Pull out the main takeaways and limitations.

```{admonition} Prereqs
:class: tip
Comfortable with gradients, Hessians, and the implicit function theorem. No deep-learning code here — everything is small enough to run in a few seconds.
```

## 1. Background: influence functions

Given a training set $\{z_1, \dots, z_n\}$ and a loss $L(z, \theta)$, the empirical risk minimiser is

$$\hat{\theta} = \arg\min_\theta \; \frac{1}{n}\sum_{i=1}^n L(z_i, \theta).$$

{cite:t}`koh2017understanding` ask: if we *upweight* a single training example $z$ by a small $\epsilon$, how does $\hat\theta$ change? A first-order Taylor expansion of the optimality condition gives

$$\left.\frac{d\hat\theta_\epsilon}{d\epsilon}\right|_{\epsilon=0} = -H_{\hat\theta}^{-1}\,\nabla_\theta L(z, \hat\theta),\qquad H_{\hat\theta} := \frac{1}{n}\sum_i \nabla^2_\theta L(z_i, \hat\theta).$$

Chain-ruling into any scalar *measurement* $f(\hat\theta)$ of the model (a loss on a test example, a class probability, whatever) yields the **influence of $z$ on $f$**:

$$\boxed{\;\mathcal{I}_f(z) \approx -\nabla_\theta f(\hat\theta)^\top H_{\hat\theta}^{-1}\,\nabla_\theta L(z, \hat\theta).\;}$$

The classical use: *attribution*. Given a behaviour you dislike, score which training documents most caused it.

## 2. The Infusion twist: run the attribution backwards

Attribution is passive — it tells you which documents matter. Infusion asks the **inverse** question:

> Can I *edit* existing training documents so that, after retraining, the model's behaviour on a chosen probe shifts in a chosen direction?

Replace $z$ with $z + \delta$ (think: swap a few tokens, nudge a few pixels). The loss gradient evaluated at the perturbed document is, to first order in $\delta$,

$$\nabla_\theta L(z+\delta, \hat\theta) \;\approx\; \nabla_\theta L(z, \hat\theta) \;+\; \bigl[\nabla_z\nabla_\theta L(z, \hat\theta)\bigr]\,\delta.$$

The object in brackets is the mixed Jacobian with shape `(|θ|, |z|)` — the sensitivity of the per-example *parameter gradient* to the *input*. Plugging this into the influence-function expression (for upweighting $\delta$ around zero) gives a predicted parameter shift

$$\Delta\hat\theta \;\approx\; -\frac{1}{n}\,H_{\hat\theta}^{-1}\,\bigl[\nabla_z\nabla_\theta L(z,\hat\theta)\bigr]\,\delta,$$

and a predicted change in the measurement

$$\Delta f(\hat\theta) \;\approx\; \nabla_\theta f(\hat\theta)^\top\,\Delta\hat\theta \;=\; \underbrace{\bigl[-\tfrac{1}{n}\nabla_\theta f(\hat\theta)^\top H_{\hat\theta}^{-1}\bigl[\nabla_z\nabla_\theta L\bigr]\bigr]}_{=:\;g(z)^\top}\,\delta.$$

So, locally, the measurement is **linear in $\delta$**, and the gradient $g(z)$ is computable. Maximising $\Delta f$ subject to a constraint $\lVert\delta\rVert \le \rho$ (or a discrete edit budget) is exactly the setting of **Projected Gradient Descent**:

```{math}
:label: eq-pgd
\delta^{(t+1)} = \mathrm{Proj}_{\mathcal{C}}\bigl(\delta^{(t)} + \eta\,g(z^{(t)})\bigr),\qquad z^{(t)} = z + \delta^{(t)}.
```

The key conceptual takeaway: **the same second-order approximation used to attribute blame becomes a first-order steering vector on the input once you linearise around $z$**. Influence functions give you the adjoint of the training-data → behaviour map, and Infusion just follows that adjoint.

### What actually makes this hard at LLM scale

Three things, roughly in order of pain:

1. **$H_{\hat\theta}^{-1}$**. For billions of parameters you cannot store it, let alone invert it. {cite:t}`grosse2023studying` use **EKFAC** {cite:p}`george2018fast` — a Kronecker-factored approximation per layer {cite:p}`martens2015optimizing` — to make the inverse-Hessian–vector product tractable.
2. **The mixed Jacobian $\nabla_z\nabla_\theta L$**. You never materialise it; you only need it via `jvp`/`vjp` with the vector coming out of the IHVP.
3. **Discrete inputs**. Text is tokens, not $\RR^d$. The paper works in the embedding space and projects back to tokens — hence the jarring but meaningful substitutions like *cat → bee*, *yarn → buzz*. (Language experiments use a small transformer on TinyStories {cite:p}`eldan2023tinystories`.)

## 3. A toy Infusion: logistic regression on synthetic data

To make the maths concrete, let's implement Infusion end-to-end on a model where *everything* is cheap: $\ell_2$-regularised logistic regression.

**Setup.**

- Data: $x_i \in \RR^d$, labels $y_i \in \{0,1\}$, $n=500$, $d=8$.
- Model: $p_\theta(y=1 \mid x) = \sigma(\theta^\top x)$.
- Loss: per-example $L(z_i,\theta) = -y_i\log p - (1-y_i)\log(1-p) + \tfrac{\lambda}{2}\lVert\theta\rVert^2$.
- Measurement: the log-probability the model assigns to class $1$ on a fixed **probe** point $x^\star$.

**Goal.** Pick a *single* training input $x_k$ and find a small perturbation $\delta$ (bounded by $\lVert\delta\rVert_2 \le \rho$) such that the retrained model's log-prob on $x^\star$ goes **up** — without changing $x_k$'s label.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
n, d = 500, 8
X = rng.standard_normal((n, d))
w_true = rng.standard_normal(d)
logits = X @ w_true
y = (rng.random(n) < 1 / (1 + np.exp(-logits))).astype(float)

x_star = rng.standard_normal(d)  # probe point
lam = 1e-2                       # l2 regularisation

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def fit(X, y, lam=lam, iters=200, lr=0.5):
    theta = np.zeros(X.shape[1])
    for _ in range(iters):
        p = sigmoid(X @ theta)
        grad = X.T @ (p - y) / len(y) + lam * theta
        theta -= lr * grad
    return theta

theta_hat = fit(X, y)
base_logprob = np.log(sigmoid(x_star @ theta_hat))
print(f"baseline log p(y=1 | x*): {base_logprob:.4f}")

### Closed forms we'll need

For logistic regression with $p_i = \sigma(\theta^\top x_i)$:

$$\nabla_\theta L(z_i, \theta) = (p_i - y_i)\,x_i + \lambda\,\theta,$$

$$H_\theta = \frac{1}{n}\sum_i p_i(1-p_i)\,x_i x_i^\top + \lambda I,$$

$$\nabla_{x_i}\nabla_\theta L(z_i,\theta) = (p_i - y_i)\,I_d + \underbrace{p_i(1-p_i)\,x_i\theta^\top}_{\text{from }\partial p_i / \partial x_i}.$$

The measurement $f(\theta) = \log \sigma(\theta^\top x^\star)$ has gradient $\nabla_\theta f(\theta) = (1 - \sigma(\theta^\top x^\star))\,x^\star$. Let's code them up.

In [ ]:
def hessian(theta, X=X, lam=lam):
    p = sigmoid(X @ theta)
    W = p * (1 - p)
    return (X.T * W) @ X / len(X) + lam * np.eye(X.shape[1])

def grad_theta_L(x, y_, theta, lam=lam):
    p = sigmoid(x @ theta)
    return (p - y_) * x + lam * theta

def mixed_jac(x, y_, theta):
    """∂/∂x of ∇_θ L(x,y;θ) — shape (|θ|, |x|)."""
    p = sigmoid(x @ theta)
    return (p - y_) * np.eye(len(x)) + p * (1 - p) * np.outer(x, theta)

def grad_theta_f(theta, x_star=x_star):
    return (1.0 - sigmoid(x_star @ theta)) * x_star

### The Infusion steering vector

Now assemble

$$g(z_k) = -\frac{1}{n}\bigl[\nabla_z\nabla_\theta L(z_k,\hat\theta)\bigr]^\top H_{\hat\theta}^{-1} \nabla_\theta f(\hat\theta).$$

Maximising $g^\top \delta$ under $\lVert\delta\rVert_2 \le \rho$ puts the optimum at $\delta^\star = \rho\,g / \lVert g\rVert$ — a one-step PGD solution (the objective is already linear).

In [ ]:
H_inv = np.linalg.inv(hessian(theta_hat))
v = H_inv @ grad_theta_f(theta_hat)        # inverse-Hessian–vector product

k = 0  # pick a training document to edit
J = mixed_jac(X[k], y[k], theta_hat)
g = -(J.T @ v) / n

rho = 0.2  # L2 budget for the perturbation
delta_star = rho * g / np.linalg.norm(g)
print(f"||delta*|| = {np.linalg.norm(delta_star):.3f}")
print(f"predicted Δ log p(y=1|x*) = {g @ delta_star:+.4f}")

### Does it actually work when we retrain?

The influence-function prediction is a **linear approximation**. Let's check it against the real thing: apply $\delta$ to $x_k$, refit from scratch, and measure.

In [ ]:
def measure(theta):
    return np.log(sigmoid(x_star @ theta))

rhos = np.linspace(0, 1.0, 21)
predicted, actual = [], []
for r in rhos:
    d_ = r * g / (np.linalg.norm(g) + 1e-12)
    predicted.append(measure(theta_hat) + g @ d_)
    X_edit = X.copy(); X_edit[k] = X[k] + d_
    predicted_minus_base = measure(fit(X_edit, y))
    actual.append(predicted_minus_base)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rhos, predicted, label="Infusion linear prediction", lw=2)
ax.plot(rhos, actual, label="retrain-from-scratch truth", lw=2, ls="--")
ax.axhline(base_logprob, color="grey", lw=1, ls=":", label="baseline")
ax.set_xlabel(r"edit budget $\rho$ ($\|\delta\|_2$)")
ax.set_ylabel(r"$\log p(y=1 \mid x^\star)$")
ax.set_title("Infusion prediction vs truth, editing one training point")
ax.legend()
fig.tight_layout();

For small $\rho$ the linear prediction tracks the true effect of retraining nearly exactly — this is what makes the attack tractable. As $\rho$ grows, the linearisation breaks down and reality deviates from prediction. That is *precisely* Insight 3 in the paper, where over-large edits on language destroy coherency before they flip predictions.

### Editing more documents

The same linear-in-$\delta$ argument extends to editing a batch of documents simultaneously: the overall measurement shift is the sum of per-document contributions (to first order), and you get a steering vector for each. Let's edit a random subset and track the effect.

In [ ]:
def infuse(indices, rho_per_doc=0.2):
    X_edit = X.copy()
    for i in indices:
        Ji = mixed_jac(X[i], y[i], theta_hat)
        gi = -(Ji.T @ v) / n
        X_edit[i] = X[i] + rho_per_doc * gi / (np.linalg.norm(gi) + 1e-12)
    return X_edit

fractions = [0.0, 0.002, 0.01, 0.05, 0.2, 0.5, 1.0]
results = []
for frac in fractions:
    m = int(frac * n)
    idx = rng.choice(n, size=m, replace=False)
    X_edit = infuse(idx)
    results.append(measure(fit(X_edit, y)) - base_logprob)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([f * 100 for f in fractions], results, marker="o", lw=2)
ax.set_xscale("symlog", linthresh=0.2)
ax.set_xlabel("% of training docs edited")
ax.set_ylabel(r"$\Delta \log p(y=1 \mid x^\star)$")
ax.set_title("Scaling the attack: more edits ⇒ bigger behavioural shift")
ax.grid(alpha=0.3)
fig.tight_layout();

Even in this toy model, editing a *fraction of a percent* of training data produces a measurable behaviour shift on the probe — while the labels are untouched and each individual perturbation is small in $\ell_2$. That's the qualitative shape of the CIFAR-10 result in the paper.

## 4. Takeaways

- Influence functions aren't just an attribution tool; they're the **adjoint of the train-data → parameters → behaviour pipeline**. Anything you can attribute with, you can in principle steer with.
- The Infusion update is *just* "IHVP, then mixed Jacobian, then project" — all three terms have standard, already-industrialised approximations (EKFAC for IHVPs, autodiff for Jacobians, discrete-PGD variants for language).
- The approximation is linear in $\delta$. This is a feature (it gives you a direct PGD objective) **and** a bug (it breaks down exactly when you try to force large behavioural changes).
- Infusion seems to *amplify behaviours the model already has* rather than install new ones. Amplifying "cat → bee" works when the bee pathway is already reachable; flipping high-confidence predictions does not.

### Open questions I want to come back to

1. **Survival under post-training**: do infused edits persist after an SFT/RLHF stage on clean data, or do they get washed out?
2. **Defender's version**: what does the symmetric defensive formulation look like? Same machinery, opposite sign, but with realistic uncertainty over which behaviours you want to protect.
3. **Scaling laws for the attack**: how does achievable $\Delta f$ scale with model size, data size, and edit budget? The paper gives datapoints; a clean theoretical bound would be nice.

```{seealso}
All citations on this page are collected in the site-wide
[References](../../references.md).
```